In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.utils.random import sample_without_replacement


In [ ]:
baseDir = '/media/austin/ThickBoy__1/DataAgression/'
pname = baseDir + 'Aggression_power.p'
cname1 = baseDir + 'Aggression_coherence1.p'
cname2 = baseDir + 'Aggression_coherence2.p'
cname3 = baseDir + 'Aggression_coherence3.p'
gname1 = baseDir + 'Aggression_granger1.p'
gname2 = baseDir + 'Aggression_granger2.p'
gname3 = baseDir + 'Aggression_granger3.p'

lname = baseDir + 'Agression_labels.p'
sname = baseDir + 'Agression_split_labels.p'

myP = pickle.load(open(pname,'rb'))
myC1 = pickle.load(open(cname1,'rb'))
myC2 = pickle.load(open(cname2,'rb'))
myC3 = pickle.load(open(cname3,'rb'))
myG1 = pickle.load(open(gname1,'rb'))
myG2 = pickle.load(open(gname2,'rb'))
myG3 = pickle.load(open(gname3,'rb'))

mySplits = pickle.load(open(sname,'rb'))
myLabels = pickle.load(open(lname,'rb'))

# Load all the labels
mouse_idx,mice = myLabels['mouse_idx'],myLabels['mice']
expD_idx,expDates= myLabels['epxD_idx'],myLabels['expDates']
group_idx,groups= myLabels['group_idx'],myLabels['groups']
condition_idx,conditions= myLabels['condition_idx'],myLabels['conditions']
behavior_idx,behaviors= myLabels['behavior_idx'],myLabels['behaviors']
behaviornon1_idx,behaviorsnon1s= myLabels['behaviornon1_idx'],myLabels['behaviorsnon1s']


In [ ]:
power = myP['power']*10
power = power.astype(np.float32)
print(np.mean(power>6))
power[power>6] = 6

C1 = myC1['coherence']
C2 = myC2['coherence']
C3 = myC3['coherence']
coherence = np.vstack((C1,C2,C3))
coherence = coherence.astype(np.float32)

G1 = myG1['granger']
G2 = myG2['granger']
G3 = myG3['granger']
granger= np.vstack((G1,G2,G3))
granger = np.exp(granger)
granger[granger>10] = 10
granger = granger.astype(np.float32)

#X = np.hstack((power,coherence))
X = np.hstack((power,coherence,granger))


In [ ]:
N = len(mouse_idx)
training_set_idx = np.ones(N)
training_set_idx[mouse_idx==mice.index('Mouse048')] = 0
training_set_idx[mouse_idx==mice.index('Mouse7980')] = 0
training_set_idx[mouse_idx==mice.index('Mouse7998')] = 0

# Divide the training and testing sets
X_train = X[training_set_idx==1]
X_test = X[training_set_idx==0]

mouse_idx_train = mouse_idx[training_set_idx==1]
mouse_idx_test = mouse_idx[training_set_idx==0]
print(np.unique(mouse_idx_test))
print(mouse_idx_test.shape)

expDate_idx_train = expD_idx[training_set_idx==1]
expDate_idx_test = expD_idx[training_set_idx==0]

group_idx_train = group_idx[training_set_idx==1]
group_idx_test = group_idx[training_set_idx==0]

condition_idx_train = condition_idx[training_set_idx==1]
condition_idx_test = condition_idx[training_set_idx==0]

behavior_idx_train = behavior_idx[training_set_idx==1]
behavior_idx_test = behavior_idx[training_set_idx==0]

aggression_idx_train = behaviornon1_idx[training_set_idx==1]
aggression_idx_test = behaviornon1_idx[training_set_idx==0]

#Numbers of observations in each set
N_train = len(mouse_idx_train)
N_test = len(mouse_idx_test)


In [ ]:
indx_pos = ((aggression_idx_train==behaviorsnon1s.index(1))&(condition_idx_train==conditions.index(4)))
indx_neg1 = ((aggression_idx_train==behaviorsnon1s.index(0))&(condition_idx_train==conditions.index(4)))
indx_neg2 = ((aggression_idx_train==behaviorsnon1s.index(0))&(condition_idx_train==conditions.index(6)))
indx_neg3 = ((aggression_idx_train==behaviorsnon1s.index(0))&(condition_idx_train==conditions.index(8)))
indx_neg = indx_neg1|indx_neg2|indx_neg3
indx_tot = indx_neg|indx_pos

In [ ]:
y_o = np.zeros(X_train.shape[0])
y_o[indx_pos] = 1

In [ ]:
X_train_new = X_train[indx_tot]
y_train_new = y_o[indx_tot]
mouse_train_new = mouse_idx_train[indx_tot]

In [ ]:
myDict = {'X_train':X_train_new,'y_train':y_train_new,'mouse_train':mouse_train_new}

In [ ]:
indx_pos = ((aggression_idx_test==behaviorsnon1s.index(1))&(condition_idx_test==conditions.index(4)))
indx_neg1 = ((aggression_idx_test==behaviorsnon1s.index(0))&(condition_idx_test==conditions.index(4)))
indx_neg2 = ((aggression_idx_test==behaviorsnon1s.index(0))&(condition_idx_test==conditions.index(6)))
indx_neg3 = ((aggression_idx_test==behaviorsnon1s.index(0))&(condition_idx_test==conditions.index(8)))
indx_neg = indx_neg1|indx_neg2|indx_neg3
indx_tot = indx_neg|indx_pos

In [ ]:
y_o = np.zeros(X_test.shape[0])
y_o[indx_pos] = 1

In [ ]:
np.sum(indx_pos)

In [ ]:
np.sum(indx_tot)

In [ ]:
X_test_new = X_test[indx_tot]
y_test_new = y_o[indx_tot]
mouse_test_new = mouse_idx_test[indx_tot]

In [ ]:
myDict['X_test'] = X_test_new
myDict['y_test'] = y_test_new
myDict['mouse_test'] = mouse_test_new

In [ ]:
pickle.dump(myDict,open('LimitedData0.p','wb'))

In [ ]:
model_LR = LR(max_iter=10000)
model_LR.fit(X_train_new,y_train_new)

In [ ]:
y_pred = model_LR.decision_function(X_train_new)
print(roc_auc_score(y_train_new,y_pred))

In [ ]:
y_pred = model_LR.decision_function(X_test_new)
print(roc_auc_score(y_test_new,y_pred))

In [ ]:
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.metrics import plot_precision_recall_curve


In [ ]:
precision,recall,_ = precision_recall_curve(y_test_new,y_pred)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(recall,precision)

In [ ]:
average_precision_score(y_test_new,y_pred)

In [ ]:
np.sum(y_pred)/len(y_pred)

In [ ]:
np.sum(y_test_new)/len(y_test_new)

In [ ]:
np.unique(mouse_test_new)

In [ ]:
np.unique(mouse_train_new)